# ArabicMedReason — Raw Inference Run (Phase 2)

This notebook runs **one model** (`Qwen/Qwen3.5-4B`) over every item in `pilot.csv`, using
**one fixed prompt template** and **greedy (deterministic) decoding**. It saves the **exact raw
model responses** for human inspection.

**This notebook does NOT:**
- grade answers or decide correctness
- compute accuracy or any other score
- use another LLM as a judge
- perform error analysis

`expected_answer` is copied into the output file only so a human reviewer can see it next to
the response. It is **never** part of the prompt, and the notebook checks this.

### Kaggle setup (before running)
1. **Settings → Accelerator:** a GPU (T4 ×1, T4 ×2 or P100).
2. **Settings → Internet:** **On** (needed for `pip` and to download the model from Hugging Face).
3. **Add Input → Upload → New Dataset:** upload `data/pilot.csv`. It will appear under
   `/kaggle/input/<your-dataset-name>/pilot.csv`. The notebook searches `/kaggle/input`
   automatically, or you can set `PILOT_CSV_OVERRIDE` below.
4. **Run All.** The output is written to `/kaggle/working/results/raw_outputs.jsonl`
   (download it from the **Output** panel and commit it to `results/` in the repo).

## 1. Environment

Kaggle images already include `torch`, `pandas` and `accelerate`. `Qwen3.5` is a recent
architecture (`model_type: qwen3_5`), so `transformers` is upgraded to a version that supports
it. `torch` is **not** reinstalled, so the CUDA build that matches Kaggle's drivers stays in place.

In [ ]:
# Upgrade transformers (and accelerate to a compatible version). Do not reinstall torch.
%pip install -q -U "transformers>=5" accelerate

> If you just upgraded `transformers`, **restart the kernel** (Run → Restart & clear outputs),
> then run all cells from the next one onward, so the new version is the one imported.

In [ ]:
import hashlib
import json
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import accelerate

print(f"Python        : {platform.python_version()}")
print(f"PyTorch       : {torch.__version__}")
print(f"Transformers  : {transformers.__version__}")
print(f"Accelerate    : {accelerate.__version__}")
print(f"Pandas        : {pd.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

GPU_INFO = []
if torch.cuda.is_available():
    print(f"CUDA (torch)  : {torch.version.cuda}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        info = {
            "index": i,
            "name": props.name,
            "total_memory_gb": round(props.total_memory / 1024**3, 2),
            "compute_capability": f"{props.major}.{props.minor}",
        }
        GPU_INFO.append(info)
        print(f"GPU {i}         : {info['name']} | {info['total_memory_gb']} GB | "
              f"compute capability {info['compute_capability']}")
else:
    print("WARNING: no GPU detected. Enable a GPU accelerator in Kaggle settings.")

## 2. Configuration

Everything that affects the run is defined here and saved with the outputs.

In [ ]:
# ---- Model ----
MODEL_NAME = "Qwen/Qwen3.5-4B"
MODEL_REVISION = None  # set to a specific commit hash to pin the exact weights (recommended once chosen)

# Qwen3.5 thinks by default (emits <think>...</think> before answering).
# Disabled here so every case gets a short, directly comparable answer, and so greedy
# decoding does not run into long reasoning loops. Recorded in run metadata.
ENABLE_THINKING = False

# ---- Generation (fixed for every case) ----
MAX_NEW_TOKENS = 512
SEED = 42

# ---- Paths ----
ON_KAGGLE = Path("/kaggle/input").exists()
PILOT_CSV_OVERRIDE = None  # e.g. "/kaggle/input/arabicmedreason-pilot/pilot.csv"

if ON_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working/results")
else:  # running locally from notebooks/
    OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_OUTPUTS_PATH = OUTPUT_DIR / "raw_outputs.jsonl"
RUN_CONFIG_PATH = OUTPUT_DIR / "run_config.json"

print(f"On Kaggle      : {ON_KAGGLE}")
print(f"Output file    : {RAW_OUTPUTS_PATH.resolve()}")
print(f"Run config file: {RUN_CONFIG_PATH.resolve()}")

In [ ]:
# Seed everything. With greedy decoding this mostly guards against hidden sampling paths;
# GPU kernels can still introduce tiny numerical differences between hardware types.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 3. Dataset loading

Kaggle mounts uploaded datasets read-only under `/kaggle/input/<dataset-name>/`. The cell below
finds `pilot.csv` there (or uses `PILOT_CSV_OVERRIDE`). The file is only read, never modified.
A SHA-256 hash of the file is recorded so each run can be traced to the exact data version.

In [ ]:
def locate_pilot_csv() -> Path:
    if PILOT_CSV_OVERRIDE:
        return Path(PILOT_CSV_OVERRIDE)
    if ON_KAGGLE:
        matches = sorted(Path("/kaggle/input").rglob("pilot.csv"))
        if not matches:
            raise FileNotFoundError(
                "pilot.csv not found under /kaggle/input. Upload data/pilot.csv as a Kaggle "
                "dataset (Add Input → Upload) or set PILOT_CSV_OVERRIDE."
            )
        if len(matches) > 1:
            print("Multiple pilot.csv files found; using the first:", *matches, sep="\n  ")
        return matches[0]
    return Path("../data/pilot.csv")


PILOT_CSV = locate_pilot_csv()
DATA_SHA256 = hashlib.sha256(PILOT_CSV.read_bytes()).hexdigest()
df = pd.read_csv(PILOT_CSV, dtype=str, keep_default_na=False, encoding="utf-8-sig")

print(f"Loaded : {PILOT_CSV}")
print(f"SHA-256: {DATA_SHA256}")

### Dataset validation

The same core checks as `src/validate_dataset.py`. The run stops if any check fails.

In [ ]:
REQUIRED_COLUMNS = ["id", "category", "language", "pair_id", "case_text", "question", "expected_answer"]
ALLOWED_CATEGORIES = {"temporal_reasoning", "contradiction_detection", "missing_evidence",
                      "false_premise", "causal_reasoning"}
ALLOWED_LANGUAGES = {"ar", "en"}

problems = []
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    problems.append(f"missing columns: {missing_cols}")
else:
    if df["id"].duplicated().any():
        problems.append(f"duplicate ids: {df.loc[df['id'].duplicated(), 'id'].tolist()}")
    for col in ["id", "pair_id", "case_text", "question", "expected_answer"]:
        if (df[col].str.strip() == "").any():
            problems.append(f"empty values in '{col}'")
    if not df["category"].isin(ALLOWED_CATEGORIES).all():
        problems.append(f"invalid categories: {sorted(set(df['category']) - ALLOWED_CATEGORIES)}")
    if not df["language"].isin(ALLOWED_LANGUAGES).all():
        problems.append(f"invalid languages: {sorted(set(df['language']) - ALLOWED_LANGUAGES)}")

assert not problems, "Dataset validation failed:\n- " + "\n- ".join(problems)

print(f"Number of cases: {len(df)}")
print(f"Number of pairs: {df['pair_id'].nunique()}")
print("\nCategory counts:")
print(df["category"].value_counts().sort_index().to_string())
print("\nLanguage counts:")
print(df["language"].value_counts().sort_index().to_string())

In [ ]:
df.head()

## 4. Model loading

`Qwen/Qwen3.5-4B` is a **vision-language** model (`Qwen3_5ForConditionalGeneration`). It is
used here with **text-only** inputs. It is loaded with the class shown on the official model
card (`AutoModelForMultimodalLM`), with `AutoModelForImageTextToText` as a fallback for
`transformers` versions that do not have that class.

**dtype:** the weights are stored in `bfloat16`.
- GPUs with compute capability ≥ 8.0 (A100, L4, …) use `bfloat16`, i.e. the native weights.
- Kaggle's **T4 (7.5) and P100 (6.0) have no native bf16**, so `float16` is used.

The weights take roughly 9–10 GB in 16-bit, which fits on a single 16 GB T4/P100.
**No quantization is used.** `device_map="auto"` spreads the model across GPUs if more than
one is available.

In [ ]:
from transformers import AutoTokenizer

try:
    from transformers import AutoModelForMultimodalLM as ModelClass
except ImportError:
    from transformers import AutoModelForImageTextToText as ModelClass

if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16
# Manual override if fp16 produces empty/garbled output or NaN errors on T4/P100:
# DTYPE = torch.float32   # ~18 GB: needs 2×T4 (device_map="auto" splits it)

print(f"Model class: {ModelClass.__name__}")
print(f"dtype      : {DTYPE}")

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
if tokenizer.chat_template is None:
    # Some multimodal repos keep the chat template only on the processor.
    from transformers import AutoProcessor
    tokenizer.chat_template = AutoProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION).chat_template

model = ModelClass.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    dtype=DTYPE,
    device_map="auto",
)
model.eval()
print(f"Loaded in {time.time() - t0:.1f}s")

# Record the exact weights commit that was downloaded.
MODEL_COMMIT = getattr(model.config, "_commit_hash", None)
print(f"Model commit: {MODEL_COMMIT}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i} memory allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")

## 5. Prompt format

Every case uses **the same single template**. The model sees only `case_text` and `question`.
The instruction text is in English for every item, so the template is identical across
languages. The model is asked to answer in the language of the case.

The template is rendered through the model's own chat template (one user turn, no system
prompt) with `enable_thinking` set as configured above. The full rendered prompt is saved for
every case.

In [ ]:
PROMPT_TEMPLATE = """You are given a synthetic clinical case and a question about it. Use only the information in the case.

Case:
{case_text}

Question:
{question}

Answer in the same language as the case, using exactly this format:
FINAL_ANSWER: <a concise answer>
REASON: <a brief explanation>"""


def build_prompt(row) -> str:
    # Only case_text and question are used. expected_answer is never passed in.
    user_message = PROMPT_TEMPLATE.format(case_text=row["case_text"], question=row["question"])
    messages = [{"role": "user", "content": user_message}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )


# Leakage checks.
# 1) Structural: the template may only use case_text and question.
import string
template_fields = {f for _, f, _, _ in string.Formatter().parse(PROMPT_TEMPLATE) if f}
assert template_fields == {"case_text", "question"}, f"unexpected template fields: {template_fields}"
# 2) Content: the expected answer must not appear in the prompt unless it is already part of
#    the case or question itself (e.g. a drug name stated in the case).
for _, row in df.iterrows():
    prompt = build_prompt(row)
    answer = row["expected_answer"].strip()
    if answer not in row["case_text"] and answer not in row["question"]:
        assert answer not in prompt, f"expected_answer leaked into prompt for {row['id']}"
print("Leakage check passed: prompts are built only from case_text and question.")

print("\n--- Example rendered prompt (first case) ---")
print(build_prompt(df.iloc[0]))

## 6. Inference

- **Greedy decoding** (`do_sample=False`, `num_beams=1`), so no sampling parameters apply.
- A fresh `GenerationConfig` is built so that sampling defaults shipped with the model
  (temperature, top_p, top_k, …) are **not** silently inherited.
- **Batch size 1**, so results do not depend on padding.
- All generation parameters are recorded in `run_config.json` and on every output row.

Note: Qwen recommends sampling for this model. Greedy decoding is used here on purpose, for
reproducibility, and this is a documented choice that may affect output quality.

In [ ]:
from transformers import GenerationConfig

eos_ids = model.generation_config.eos_token_id
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else (
    eos_ids[0] if isinstance(eos_ids, list) else eos_ids)

GEN_PARAMS = {
    "do_sample": False,
    "num_beams": 1,
    "max_new_tokens": MAX_NEW_TOKENS,
    "repetition_penalty": 1.0,
    "eos_token_id": eos_ids,
    "pad_token_id": pad_id,
}
GEN_CONFIG = GenerationConfig(**GEN_PARAMS)
print(json.dumps(GEN_PARAMS, indent=2))

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

RUN_CONFIG = {
    "run_id": RUN_ID,
    "model_name": MODEL_NAME,
    "model_revision_requested": MODEL_REVISION,
    "model_commit": MODEL_COMMIT,
    "model_class": ModelClass.__name__,
    "dtype": str(DTYPE),
    "quantization": None,
    "enable_thinking": ENABLE_THINKING,
    "generation_params": GEN_PARAMS,
    "seed": SEED,
    "batch_size": 1,
    "prompt_template": PROMPT_TEMPLATE,
    "data_file": str(PILOT_CSV),
    "data_sha256": DATA_SHA256,
    "n_cases": len(df),
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "accelerate": accelerate.__version__,
        "cuda": torch.version.cuda,
        "gpus": GPU_INFO,
        "on_kaggle": ON_KAGGLE,
    },
}
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved run config to {RUN_CONFIG_PATH}")

In [ ]:
input_device = next(model.parameters()).device
records = []

for i, row in df.iterrows():
    prompt = build_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(input_device)
    prompt_len = inputs["input_ids"].shape[1]

    t0 = time.time()
    with torch.inference_mode():
        output_ids = model.generate(**inputs, generation_config=GEN_CONFIG)
    elapsed = time.time() - t0

    new_ids = output_ids[0, prompt_len:].tolist()
    # Exact decoded response, special tokens included (e.g. the end-of-turn token).
    raw_output = tokenizer.decode(new_ids, skip_special_tokens=False)
    # Same response with special tokens removed, for easier reading. Not otherwise cleaned.
    raw_output_no_special = tokenizer.decode(new_ids, skip_special_tokens=True)
    eos_set = set(eos_ids if isinstance(eos_ids, list) else [eos_ids])
    stopped_on_eos = len(new_ids) > 0 and new_ids[-1] in eos_set

    records.append({
        "id": row["id"],
        "pair_id": row["pair_id"],
        "category": row["category"],
        "language": row["language"],
        "case_text": row["case_text"],
        "question": row["question"],
        "expected_answer": row["expected_answer"],  # for human review only; never in the prompt
        "raw_model_output": raw_output,
        "raw_model_output_no_special_tokens": raw_output_no_special,
        "output_token_ids": new_ids,
        "n_prompt_tokens": prompt_len,
        "n_output_tokens": len(new_ids),
        "stopped_on_eos": stopped_on_eos,       # False => hit max_new_tokens (truncated)
        "generation_seconds": round(elapsed, 2),
        "rendered_prompt": prompt,
        "model_name": MODEL_NAME,
        "model_commit": MODEL_COMMIT,
        "dtype": str(DTYPE),
        "enable_thinking": ENABLE_THINKING,
        "generation_params": GEN_PARAMS,
        "run_id": RUN_ID,
        "data_sha256": DATA_SHA256,
    })
    flag = "" if stopped_on_eos else "  [TRUNCATED at max_new_tokens]"
    print(f"[{len(records)}/{len(df)}] {row['id']}: {len(new_ids)} tokens, {elapsed:.1f}s{flag}")

print("Inference complete.")

## 7. Saving raw outputs

One JSON object per line, UTF-8, with Arabic kept as-is (`ensure_ascii=False`). The file is written
to `/kaggle/working/results/raw_outputs.jsonl` on Kaggle (`../results/raw_outputs.jsonl` locally).
A re-run overwrites it, so download it after each run.

In [ ]:
with RAW_OUTPUTS_PATH.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Read back to confirm the file is complete and valid.
with RAW_OUTPUTS_PATH.open(encoding="utf-8") as f:
    reloaded = [json.loads(line) for line in f]
assert len(reloaded) == len(df), "Row count mismatch between dataset and saved outputs"
assert [r["id"] for r in reloaded] == df["id"].tolist()

print(f"Saved {len(reloaded)} records to {RAW_OUTPUTS_PATH.resolve()}")
print(f"Run config: {RUN_CONFIG_PATH.resolve()}")

### Side-by-side view for human inspection

The table only displays the outputs. **No grading or scoring is performed.**

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.DataFrame(reloaded)[["id", "question", "expected_answer", "raw_model_output_no_special_tokens", "stopped_on_eos"]]

---
**End of Phase 2 notebook.** Next steps (manual): download `raw_outputs.jsonl` and
`run_config.json` from the Kaggle **Output** panel, commit them to `results/`, and inspect the
responses by hand. Grading rubric and analysis come in a later phase.